<span style="color:lightgreen">

# Notebook extra 2.1

En este notebook se experimenta con los parámetros de transcripción con traducción de whisper en cascada.

En este ejercicio exploramos hiperparámetros de transcripción para ST en cascada

Covost2 pt-en

</span>

# ASR Baseline experiment using Whisper and Europarl-ST (Spanish to English)

In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for ASR on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using the Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing Word Error Rate, WER)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

import jiwer

model = whisper.load_model("base")

Load Europarl-ST speech dataset

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)
data = raw_datasets["test"]
data_train = raw_datasets["train"]

print(data)

Dataset({
    features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
    num_rows: 4023
})


In [4]:

experiments = [
    {
        "name": "Experiment 1: Default parameters",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "Este é um discurso do Parlamento Europeu sobre política e legislação."
        }
    },

    # WITH NO NORMALIZATION
    {
        "name": "Experiment 1: Default parameters",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "Este é um discurso do Parlamento Europeu sobre política e legislação."
        }
    }
]

# Display experiments
for exp in experiments:
    print(f"\n{exp['name']}")
    for key, value in exp['params'].items():
        print(f"  {key}: {value}")



Experiment 1: Default parameters
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 2: Low temperature (more deterministic)
  temperature: 0.0
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 3: Lenient thresholds
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 3.0
  logprob_threshold: -1.5
  no_speech_threshold: 0.8
  condition_on_previous_text: True
  initial_prompt: None

Experiment 4: No conditioning on previous text
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: False
  initial_prompt: None

Experiment 5: With initial prompt and strict thresholds
  temperature: 0.0
  compression_

<span style="color:lightgreen">
mute warnings

<span>

In [5]:
import torch
import warnings
import os
import logging

# Set Tensor Core precision for RTX 5070
torch.set_float32_matmul_precision('high')

# Suppress PyTorch Lightning tips and info messages
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

# Alternative: suppress all Lightning logs
# os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'

Transcribe all the audio data using the Whisper (base) model. The ASR output is stored in hypotheses. At the same time, transcription references are stored into references and translation references into translations.

In [ ]:
from maikol_utils.print_utils import print_separator, print_color
normalizer = BasicTextNormalizer()

experiments_results = {}

for exp in experiments:
    print_separator(exp["name"])

    experiments_results[exp["name"]] = {}
    hypotheses = []

    for sample in tqdm(data["file"]):
        hypotheses.append((model.transcribe(
            sample, 
            language=CONFIG.src_name,
            **exp["params"]
        ))['text'])

    data_df = pd.DataFrame(dict(hypothesis=hypotheses, reference=data["sentence"], translation=data["translation"]))

    if exp["normalize"]:
        data_df["hypothesis_clean"] = [normalizer(text) for text in data_df["hypothesis"]]
        data_df["reference_clean"] = [normalizer(text) for text in data_df["reference"]]
        data_df["translation_clean"] = [normalizer(text) for text in data_df["translation"]]
    else:
        data_df["hypothesis_clean"] = data_df["hypothesis"]
        data_df["reference_clean"] = data_df["reference"]
        data_df["translation_clean"] = data_df["translation"]

    wer = jiwer.wer(list(data_df["reference_clean"]), list(data_df["hypothesis_clean"]))

    experiments_results[exp["name"]]["hypotheses"] = hypotheses
    experiments_results[exp["name"]]["translations"] = data_df["translation_clean"]
    experiments_results[exp["name"]]["references"] = data_df["reference_clean"]
    experiments_results[exp["name"]]["sources"] = data_df["hypothesis_clean"]
    experiments_results[exp["name"]]["wer"] = wer

    print_color(f"WER: {wer:.2%}", color="cyan")


________________________________________________________________
                Experiment 1: Default parameters                



  0%|          | 0/4023 [00:00<?, ?it/s]

WER: 24.06%
________________________________________________________________
       Experiment 2: Low temperature (more deterministic)       



  0%|          | 0/4023 [00:00<?, ?it/s]

WER: 23.63%
________________________________________________________________
                Experiment 3: Lenient thresholds                



...

WER: 29.09%
________________________________________________________________
         Experiment 4: No conditioning on previous text         



  0%|          | 0/4023 [00:00<?, ?it/s]

WER: 29.30%
________________________________________________________________
    Experiment 5: With initial prompt and strict thresholds     



  0%|          | 0/4023 [00:00<?, ?it/s]

WER: 29.23%


<p style="page-break-after:always;"></p>

<span style="color:lightgreen">

### Best scores

<span>

In [7]:
best_wer_exp = min(experiments_results.items(), key=lambda item: item[1]['wer'])
best_wer_score = best_wer_exp[1]['wer']

print_separator("Best Experiment by WER Score")
print_color(f'Best Experiment: {best_wer_exp[0]} with WER score: {best_wer_score:.2%}', color="green")


________________________________________________________________
                  Best Experiment by WER Score                  

Best Experiment: Experiment 3: Lenient thresholds with WER score: 29.09%


'\x1bBest Experiment: Experiment 3: Lenient thresholds with WER score: 29.09%\x1b'

Transcription hypotheses, references and translations are stored into a Pandas dataframe. Show the two first hypotheses.

In [8]:
data = pd.DataFrame(dict(hypothesis=experiments_results[best_wer_exp[0]]["hypotheses"], reference=experiments_results[best_wer_exp[0]]["references"], translation=experiments_results[best_wer_exp[0]]["translations"]))
pd.set_option('display.max_colwidth', None)
data.head(2)

,hypothesis,reference,translation
0,e dia de ir emprestado as pessoas daudei.,Pedir dinheiro emprestado às pessoas da aldeia,Borrow money from people in the village
1,Tronca-vos.,Trancá-los,Lock them up


Hypotheses and references are normalized using the Whisper Basic text standardisation/normalization module

In [9]:
data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["translation_clean"] = [normalizer(text) for text in data["translation"]]

All the data is stored into a file using 'csv' format

In [10]:
data.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en.csv'), encoding='utf-8')

# Ahora generamos datos para poder finetunear los modelos

In [ ]:
for exp in experiments:
    if exp["name"] == best_wer_exp[0]:
        best_exp_config = exp
        break

hypotheses = []

for sample in tqdm(data_train["file"]):
    hypotheses.append((model.transcribe(
        sample, 
        language=CONFIG.src_name,
        **best_exp_config["params"]
    ))['text'])

data_df = pd.DataFrame(dict(hypothesis=hypotheses, reference=data_train["sentence"], translation=data_train["translation"]))

if best_exp_config["normalize"]:
    data_df["hypothesis_clean"] = [normalizer(text) for text in data_df["hypothesis"]]
    data_df["reference_clean"] = [normalizer(text) for text in data_df["reference"]]
    data_df["translation_clean"] = [normalizer(text) for text in data_df["translation"]]
else:
    data_df["hypothesis_clean"] = data_df["hypothesis"]
    data_df["reference_clean"] = data_df["reference"]
    data_df["translation_clean"] = data_df["translation"]

wer = jiwer.wer(list(data_df["reference_clean"]), list(data_df["hypothesis_clean"]))

data_df["hypotheses"] = hypotheses
data_df["translations"] = data_df["translation_clean"]
data_df["references"] = data_df["reference_clean"]
data_df["sources"] = data_df["hypothesis_clean"]
data_df["wer"] = wer

print_color(f"WER: {wer:.2%}", color="cyan")



data_df.to_csv(os.path.join(CONFIG.RESULTS_PATH, 'Extra2.1-L4.1_ASR_Whisper_Baseline_Covost2_pt-en-train.csv'), encoding='utf-8')
